#### Step 1 导入相关包

In [ ]:
import pandas as pd
from collections import defaultdict,Counter
from tqdm import tqdm
import pickle

import matplotlib.pyplot as plt
# 设置全局字体
plt.rcParams['font.family'] = 'Times New Roman'

import numpy as np
from scipy.interpolate import interp1d

#### Step 加载相关数据

In [ ]:
with open('data/id2_affiliation_entity.pickle','rb') as file:
    id2_affiliation_entity = pickle.load(file)

#### Step 2 读取46w条 excel 数据

In [ ]:
df_reference = pd.read_excel('data/total_reference.xlsx')
df_reference.dropna()
df_reference = df_reference.drop_duplicates()
df_reference

#### Step 3 提取数据

In [ ]:
map_dict = {
    'education':'academic',
    'facility':'other',
    'government': 'other',
    'nonprofit': 'other',
    'company':'industry',
    'healthcare': 'academic',
}

In [ ]:
def check_categories(institution_type_lists):
    if 'academic' in institution_type_lists and 'industry' in institution_type_lists:
        return 'cooperation'
    else:
        return institution_type_lists[0] 

def process_affiliation(institution_type_lists,mode ='first'):
    if mode == 'first':
        return institution_type_lists[0]
    
    elif mode == 'all':
        return check_categories(institution_type_lists)
    
    elif mode == 'first and last':
        return institution_type_lists[0],institution_type_lists[-1]

In [ ]:
paper_reference_dict = defaultdict(list)

# 定义一个获取默认值的函数
def get_mapped_value(item):
    return map_dict.get(item, 'other')  # 如果 item 不在 map_dict 中，返回 None

for row in tqdm(df_reference.iterrows(),total=len(df_reference)):
    paper_id = row[1][0]
    title = row[1][1]
    try:
        institution_type_lists = eval(row[1][3])
    except Exception as e:
        print(row[0],row[1][3])
        continue
    if institution_type_lists == []:
        continue
    mapped_institution_type_lists = list(map(get_mapped_value,institution_type_lists))
    insti_type = process_affiliation(mapped_institution_type_lists,mode='all')
    paper_reference_dict[paper_id].append(insti_type)
    
print('数据解析正确')

In [ ]:
cite_year_dict = defaultdict(dict)
for year in range(2000,2023):
    cite_year_dict[year] = defaultdict(list)
for paper_id in tqdm(paper_reference_dict):
    information_dict = id2_affiliation_entity[paper_id]
    try:
        publish_year = int(information_dict['year'])
        institution_type = information_dict['Cooperation type']
    except Exception as e:
        pass

    cite_year_dict[publish_year][institution_type].append(paper_reference_dict[paper_id])

In [ ]:
count_dict = defaultdict(dict)
for year in range(2000,2023):
    count_dict[year] = defaultdict(dict)
    for institution_type in ['academic','industry','cooperation']:
        count_dict[year][institution_type] = dict()
        for sub_lists in cite_year_dict[year][institution_type]:
            for item in sub_lists:
                
                count_dict[year][institution_type][item] = count_dict[year][institution_type].get(item,0) + 1

In [ ]:
count_dict

In [ ]:
count_dict[2000]

In [ ]:
# 每一年构造一个流动关系
interval_year = 1
flow_lists = defaultdict(dict)
for i in range(2000, 2023, interval_year):
    temp_dict = defaultdict(dict)

    for j in range(i, i+interval_year):
        if j == 2023:
            break
        
        for institution_type in ['academic','industry','cooperation']:
            temp_dict[institution_type] = {each: temp_dict[institution_type].get(each, 0) + count_dict[j][institution_type].get(each, 0) for each in (*temp_dict[institution_type], *count_dict[j][institution_type])}
    flow_lists[i] = temp_dict

#### Step 3 算工业界引用占比的变化

In [ ]:
flow_lists

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

years = range(2000, 2023)
# 设置全局字体大小
plt.rcParams.update({'font.size': 25})
# 要分析的机构类型
analysis_institutions = ['academic', 'industry', 'cooperation', 'other']

# 循环绘制每个机构类型的数据
for analysis_institution in analysis_institutions:
    # 创建新的图形
    plt.figure(figsize=(12,8), dpi=300)
    
    # 定义空列表存储每年的占比数据
    ratio_lists = []
    
    for institution_type in ['academic', 'industry', 'cooperation']:
        ratio_list = []
        
        for year in years:
            analysis_dict = flow_lists[year][institution_type]
            # 计算所有值的总和
            total_sum = sum(analysis_dict.values())
            ratio = analysis_dict[analysis_institution] / total_sum
            ratio_list.append(ratio)
        
        ratio_lists.append(ratio_list)
    
    # 绘制折线图
    for i, institution_type in enumerate(['academic', 'industry', 'cooperation']):
        plt.plot(years, ratio_lists[i], marker='o', label=institution_type,linewidth=3)
    
    # plt.title(f'Citation belongs to {analysis_institution}', fontsize=18)
    plt.xlabel('Year', fontsize=30)
    plt.ylabel('Ratio', fontsize=30)
    # plt.legend(loc='upper center', bbox_to_anchor=(0.5, 1), ncol=3, fontsize=20)
    plt.grid(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 获取tab10颜色循环的颜色
colors = plt.cm.tab10.colors

# 创建一个图例显示四种颜色的横线，线宽为3
legend_labels = ['academic', 'industry', 'cooperation']
legend_lines = [plt.Line2D([0], [1], color=colors[i], lw=3) for i in range(len(colors))]

plt.figure(figsize=(6, 0.5), dpi=300)
plt.legend(legend_lines, legend_labels, loc='center', ncol=len(colors),fontsize=10)
plt.axis('off')  # 关闭坐标轴显示，仅显示图例
plt.show()
